# Conditional Probability — Notebook

Companion to `README.md` and `lesson.tex`. Stdlib only.

We work through:
1. The definition $P(A\mid B)=P(A\cap B)/P(B)$ on a finite sample space.
2. Bayes' theorem (medical test): formula + Monte Carlo agreement.
3. Law of total probability sanity check.
4. Monty Hall: switching wins ~2/3.
5. Chain rule via a small Markov chain.


In [ ]:
# Cell 1: definition on a finite space
# Omega = pairs (d1, d2) from two fair dice. Use only `random` + comprehensions.
from fractions import Fraction

Omega = [(i, j) for i in range(1, 7) for j in range(1, 7)]
def P(event):
    return Fraction(sum(1 for w in Omega if event(w)), len(Omega))

A = lambda w: w[0] + w[1] == 8           # sum is 8
B = lambda w: w[0] == 3                  # first die is 3
PA, PB = P(A), P(B)
PAB = P(lambda w: A(w) and B(w))
print('P(A) =', PA, '  P(B) =', PB, '  P(A&B) =', PAB)
print('P(A|B) =', PAB / PB, '  (expected 1/6 since only (3,5) makes 8 given d1=3)')


In [ ]:
# Cell 2: Bayes on a medical test (analytic)
prior, sens, spec = 0.01, 0.99, 0.95
p_pos = sens * prior + (1 - spec) * (1 - prior)
post = sens * prior / p_pos
print(f'P(positive) = {p_pos:.6f}')
print(f'P(disease | positive) = {post:.6f}')
print('Even with a 99% sensitive test, posterior is ~16.7% because prior is tiny.')


In [ ]:
# Cell 3: Bayes Monte Carlo cross-check
import random
rng = random.Random(42)
N = 200_000
pos = 0
pos_and_sick = 0
for _ in range(N):
    sick = rng.random() < prior
    positive = rng.random() < (sens if sick else 1 - spec)
    if positive:
        pos += 1
        if sick:
            pos_and_sick += 1
mc = pos_and_sick / pos
print(f'Monte Carlo P(sick | positive) = {mc:.6f}  (analytic {post:.6f})')
assert abs(mc - post) < 0.01


In [ ]:
# Cell 4: law of total probability
# P(positive) reconstructed from a partition {sick, healthy}.
lhs = p_pos
rhs = sens * prior + (1 - spec) * (1 - prior)
print('LHS  P(+)            =', lhs)
print('RHS  sum_i P(+|A_i)P(A_i) =', rhs)
assert abs(lhs - rhs) < 1e-12


In [ ]:
# Cell 5: Monty Hall — switching wins 2/3
rng = random.Random(7)
trials = 10_000
stay_wins = switch_wins = 0
for _ in range(trials):
    car = rng.randrange(3)
    pick = rng.randrange(3)
    cands = [d for d in range(3) if d != pick and d != car]
    opened = rng.choice(cands)
    switch_to = next(d for d in range(3) if d != pick and d != opened)
    stay_wins += (pick == car)
    switch_wins += (switch_to == car)
print(f'P(win | stay)   ≈ {stay_wins/trials:.4f}  (theory 1/3)')
print(f'P(win | switch) ≈ {switch_wins/trials:.4f}  (theory 2/3)')


In [ ]:
# Cell 6: chain rule via a 3-step Markov chain
# Transition matrix on states {0, 1}.
T = {0: {0: 0.7, 1: 0.3}, 1: {0: 0.4, 1: 0.6}}
pi0 = {0: 0.5, 1: 0.5}
# P(X0=0, X1=1, X2=0) by chain rule:
p_chain = pi0[0] * T[0][1] * T[1][0]
print('P(X0=0, X1=1, X2=0) =', p_chain)
# Cross-check by enumerating the joint over (X0, X1, X2).
joint = 0.0
for x0 in (0, 1):
    for x1 in (0, 1):
        for x2 in (0, 1):
            if (x0, x1, x2) == (0, 1, 0):
                joint += pi0[x0] * T[x0][x1] * T[x1][x2]
print('Direct enumeration  =', joint)
assert abs(p_chain - joint) < 1e-12


## Takeaways

- Conditional probability is a *rescaling*: drop mass outside $B$, renormalize.
- Bayes inverts the conditional direction; total probability supplies the denominator.
- Monty Hall is a one-line Bayes computation; the simulation just confirms the formula.
- The chain rule is what makes autoregressive models tractable.
